In [ ]:
%matplotlib widget

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import flammkuchen as fl
from pathlib import Path
import tifffile
from scipy.signal import find_peaks

In [ ]:
def find_trial_start_times(regressor, min_gap=30):
    """
    Find the start times of stimulus trials.
    
    Parameters:
    -----------
    regressor : numpy.ndarray
        Binary regressor (0/1) indicating when stimulus is present
    min_gap : int
        Minimum number of frames between trials
    
    Returns:
    --------
    list
        Indices where trials start
    """
    # Find transitions from 0 to 1
    transitions = np.where(np.diff(regressor.astype(int)) == 1)[0] + 1
    
    # Filter out transitions that are too close to previous transition
    trial_starts = [transitions[0]]
    for t in transitions[1:]:
        if t - trial_starts[-1] >= min_gap:
            trial_starts.append(t)
    
    return trial_starts

def extract_trial_responses(roi_trace, regressor, pre_frames=30, post_frames=60):
    """
    Extract trial-by-trial responses for an ROI.
    
    Parameters:
    -----------
    roi_trace : numpy.ndarray
        Time series of ΔF/F for a single ROI
    regressor : numpy.ndarray
        Binary regressor (0/1) indicating when stimulus is present
    pre_frames : int
        Number of frames to include before trial start
    post_frames : int
        Number of frames to include after trial start
    
    Returns:
    --------
    numpy.ndarray
        Array of shape (n_trials, pre_frames+post_frames) containing trial responses
    """
    # Find trial start times
    trial_starts = find_trial_start_times(regressor)
    
    # Initialize array to store responses
    n_trials = len(trial_starts)
    trial_length = pre_frames + post_frames
    responses = np.zeros((n_trials, trial_length))
    
    # Extract response for each trial
    for i, start in enumerate(trial_starts):
        # Check if we have enough frames before and after
        if start >= pre_frames and start + post_frames <= len(roi_trace):
            trial_slice = slice(start - pre_frames, start + post_frames)
            responses[i, :] = roi_trace[trial_slice]
        else:
            # If trial is at the boundary, fill with NaNs
            responses[i, :] = np.nan
    
    # Filter out trials with NaNs
    responses = responses[~np.isnan(responses).any(axis=1)]
    
    return responses

def plot_roi_motion_responses(neural_data, regressors, roi_index, imaging_rate=3.0, 
                            pre_seconds=10, post_seconds=20, 
                            output_dir=None, session_name="", filename=None):
    """
    Generate response plots for a specific ROI.
    
    Parameters:
    -----------
    neural_data : numpy.ndarray
        Neural activity data (ROIs x time points)
    regressors : dict
        Dictionary containing 'left_regressor' and 'right_regressor'
    roi_index : int
        Index of the ROI to plot
    imaging_rate : float
        Imaging frame rate in Hz
    pre_seconds : float
        Number of seconds to include before trial start
    post_seconds : float
        Number of seconds to include after trial start
    output_dir : str
        Directory to save output figure
    session_name : str
        Name of the session for the plot title
    filename : str
        Output filename (default: roi_{roi_index}_responses.png)
    
    Returns:
    --------
    str
        Path to saved figure
    """
    print(f"\nAnalyzing ROI {roi_index}")
    
    # Set output paths
    if output_dir is None:
        output_dir = os.getcwd()
    
    if filename is None:
        filename = f"roi_{roi_index}_motion_responses.png"
    
    output_path = os.path.join(output_dir, filename)
    
    # Convert time to frames
    pre_frames = int(pre_seconds * imaging_rate)
    post_frames = int(post_seconds * imaging_rate)
    
    try:
        # Extract regressors
        left_regressor = regressors['left_regressor']
        right_regressor = regressors['right_regressor']
        
        # Extract ROI trace from neural data
        if neural_data.shape[0] <= roi_index:
            print(f"ROI index {roi_index} is out of bounds for neural data with {neural_data.shape[0]} ROIs")
            return None
        
        roi_trace = neural_data[roi_index]
        
        # Extract trial responses
        left_responses = extract_trial_responses(roi_trace, left_regressor, 
                                               pre_frames=pre_frames, post_frames=post_frames)
        right_responses = extract_trial_responses(roi_trace, right_regressor, 
                                                pre_frames=pre_frames, post_frames=post_frames)
        
        # Create time vector for x-axis
        time_vector = np.arange(-pre_frames, post_frames) / imaging_rate
        
        # Create figure
        fig, axes = plt.subplots(1, 2, figsize=(10, 6), sharey=True)
        
        # Plot right motion responses (in red)
        if len(right_responses) > 0:
            # Average of first 9 trials (or fewer if less available)
            n_avg_trials = min(9, len(right_responses) - 1)
            if n_avg_trials > 0:
                avg_response = np.mean(right_responses[:n_avg_trials], axis=0)
                std_response = np.std(right_responses[:n_avg_trials], axis=0) / np.sqrt(n_avg_trials)
                axes[0].fill_between(time_vector, avg_response - std_response, 
                                   avg_response + std_response, 
                                   color='red', alpha=0.3)
                axes[0].plot(time_vector, avg_response, color='red', alpha=0.7, 
                           label=f'Average (first {n_avg_trials} trials)')
            
            # Last trial
            if len(right_responses) > 0:
                axes[0].plot(time_vector, right_responses[-1], color='darkred', linewidth=2,
                           label='Last trial')
            
            # Add vertical line at stimulus onset
            axes[0].axvline(x=0, color='k', linestyle='--', alpha=0.5)
            
            # Add horizontal line at y=0
            axes[0].axhline(y=0, color='k', linestyle='-', alpha=0.3)
            
            # Shade the stimulus presentation period (assuming 5s duration)
            axes[0].axvspan(0, 5, alpha=0.2, color='gray')
            
            axes[0].set_title(f"Response to Rightward Motion (n={len(right_responses)} trials)")
        else:
            axes[0].text(0.5, 0.5, "No rightward motion trials found", 
                       ha='center', va='center', transform=axes[0].transAxes)
            axes[0].set_title("Response to Rightward Motion")
        
        # Plot left motion responses (in blue)
        if len(left_responses) > 0:
            # Average of first 9 trials (or fewer if less available)
            n_avg_trials = min(9, len(left_responses) - 1)
            if n_avg_trials > 0:
                avg_response = np.mean(left_responses[:n_avg_trials], axis=0)
                std_response = np.std(left_responses[:n_avg_trials], axis=0) / np.sqrt(n_avg_trials)
                axes[1].fill_between(time_vector, avg_response - std_response, 
                                   avg_response + std_response, 
                                   color='blue', alpha=0.3)
                axes[1].plot(time_vector, avg_response, color='blue', alpha=0.7, 
                           label=f'Average (first {n_avg_trials} trials)')
            
            # Last trial
            if len(left_responses) > 0:
                axes[1].plot(time_vector, left_responses[-1], color='darkblue', linewidth=2,
                           label='Last trial')
            
            # Add vertical line at stimulus onset
            axes[1].axvline(x=0, color='k', linestyle='--', alpha=0.5)
            
            # Add horizontal line at y=0
            axes[1].axhline(y=0, color='k', linestyle='-', alpha=0.3)
            
            # Shade the stimulus presentation period (assuming 5s duration)
            axes[1].axvspan(0, 5, alpha=0.2, color='gray')
            
            axes[1].set_title(f"Response to Leftward Motion (n={len(left_responses)} trials)")
        else:
            axes[1].text(0.5, 0.5, "No leftward motion trials found", 
                       ha='center', va='center', transform=axes[1].transAxes)
            axes[1].set_title("Response to Leftward Motion")
        
        # Set axis labels
        for ax in axes:
            ax.set_xlabel("Time from stimulus onset (s)")
            ax.legend()
            ax.grid(True, alpha=0.3)
        
        axes[0].set_ylabel("ΔF/F")
        
        # Set title
        if session_name:
            plt.suptitle(f"ROI {roi_index} Responses to Motion - Session {session_name}", fontsize=14)
        else:
            plt.suptitle(f"ROI {roi_index} Responses to Motion", fontsize=14)
        
        plt.tight_layout()
        plt.savefig(output_path, dpi=150)
        plt.close()
        
        print(f"Saved response plot to {os.path.basename(output_path)}")
        
        return output_path
        
    except Exception as e:
        print(f"Error generating ROI response plot: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

def load_session_data(session_folder):
    """
    Load regressors and neural data from a session folder.
    
    Parameters:
    -----------
    session_folder : str
        Path to the session folder
    
    Returns:
    --------
    tuple
        (regressors, neural_data, session_name)
    """
    session_name = os.path.basename(session_folder)
    print(f"Loading data from session: {session_name}")
    
    # Load regressors
    regressor_path = os.path.join(session_folder, "motion_regressors.h5")
    if not os.path.exists(regressor_path):
        print(f"Regressor file not found: {regressor_path}")
        return None, None, session_name
    
    regressors = fl.load(regressor_path)
    
    # Load neural data - this is a placeholder since you'll pass this directly
    # In a real scenario, you might load from DFF TIFFs or ROI trace files
    neural_data = None
    
    return regressors, neural_data, session_name

def process_roi_batch(neural_data, regressors, roi_indices, output_dir=".", session_name=""):
    """
    Process multiple ROIs for a given neural dataset.
    
    Parameters:
    -----------
    neural_data : numpy.ndarray
        Neural activity data (ROIs x time points)
    regressors : dict
        Dictionary containing 'left_regressor' and 'right_regressor'
    roi_indices : list
        List of ROI indices to process
    output_dir : str
        Directory to save output figures
    session_name : str
        Name of the session for the plot titles
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Process each ROI
    for roi_idx in roi_indices:
        plot_roi_motion_responses(
            neural_data=neural_data,
            regressors=regressors,
            roi_index=roi_idx,
            imaging_rate=3.0,  # Assuming 3 Hz imaging rate
            pre_seconds=10,    # 10 seconds before stimulus
            post_seconds=20,   # 20 seconds after stimulus
            output_dir=output_dir,
            session_name=session_name
        )



In [ ]:
from glob import glob

In [ ]:
master = Path(r"Z:\Hagar\main\e0020 imaging")

fish_list = list(master.glob("*_v41*"))
fish = fish_list[0]
print(fish)
num_fish = len(fish_list)

In [ ]:
base_dir = str(fish / 'suite2p' / '0000')
traces = fl.load(Path(base_dir) / 'data_from_suite2p_cells.h5')['traces']
regressors = fl.load(Path(base_dir) /  "motion_regressors.h5")

In [ ]:
np.shape(traces)

In [ ]:
plot_roi_motion_responses(
    neural_data=traces,
    regressors=regressors,
    roi_index=145,  # Change to the ROI you want to analyze
    output_dir="./output_plots",
    session_name="Session_0000"
)
